## Giới thiệu
`MaskedLMs.ipynb` là Demo xây dựng một **bidirectional transformer encoder** (kiến trúc Encoder-only) phục vụ cho bài toán **masked language modeling (MLM)**. Khác với các mô hình causal (chỉ nhìn từ trái sang phải), mô hình này cho phép học ngữ cảnh từ cả hai phía để hiểu sâu hơn về ý nghĩa của từ.

## Mô hình
- Sử dụng **word-level tokenization**
- Khởi tạo embedding từ vựng (**300d**)
- Kiến trúc **Bidirectional Transformer Encoder** (tương tự BERT)
- Gồm:
  - **Token Embedding**
  - **Positional Embedding**
  - **Multi-Head Self-Attention** (không sử dụng causal mask, cho phép nhìn toàn bộ câu)
  - **Feed Forward Network** (kết hợp **Pre-LayerNorm** để tối ưu hóa quá trình hội tụ)
  - **Linear LM Head**
- Huấn luyện theo bài toán **cloze task** (điền từ vào chỗ trống)

## Mục tiêu
Giúp hiểu pipeline của một mô hình học biểu diễn ngôn ngữ (Representation Model) từ:

**dữ liệu → chunking → masking → embedding → bidirectional self-attention → huấn luyện → điền từ vào chỗ trống**

## Dataset
- Sử dụng **Tiny Shakespeare** làm tập dữ liệu huấn luyện
- Đây là một corpus văn bản nhỏ, phù hợp để demo khả năng học ngữ pháp và mối liên hệ từ vựng trong các đoạn kịch

## Data Preprocessing
- Đọc dữ liệu văn bản từ file `input.txt`
- Gom văn bản thành các đoạn có độ dài cố định (**chunking**) là **64 tokens**
- Áp dụng **15% masking logic** trên mỗi đoạn văn bản theo chuẩn BERT:
  - **80%** trường hợp thay bằng token **[MASK]**
  - **10%** trường hợp thay bằng một từ ngẫu nhiên trong từ điển
  - **10%** trường hợp giữ nguyên từ gốc để mô hình học cách đại diện cho từ thực
- Khởi tạo tập **special tokens** gồm:
  - **pad**
  - **unk**
  - **[MASK]**
  - **[CLS]**
  - **[SEP]**
- Xây dựng **vocabulary** đầy đủ để triệt tiêu lỗi **<unk>** trong tập huấn luyện

## Training Configuration
- **Model:** Bidirectional Transformer Encoder
- **Embedding dimension:** `300`
- **Number of heads:** `6`
- **Number of layers:** `4`
- **Dropout:** `0.1`
- **Optimizer:** AdamW
- **Learning rate:** `3 × 10^-4`
- **Epochs:** `100`
- **Objective:** Masked language modeling (MLM)
- **Loss:** Cross entropy loss (chỉ tính tại các vị trí bị mask thông qua **ignore_index=-100**)

In [ ]:
# Tải dữ liệu Tiny Shakespeare
!wget -O input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-04-20 13:52:18--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-04-20 13:52:18 (37.3 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
# =========================
# 1. LOAD DATA & CHUNKING
# =========================

import torch
import torch.nn as nn
import torch.optim as optim
import math
import random
from collections import Counter

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

words = text.split()

SEQ_LEN = 64
corpus = []

for i in range(0, len(words) - SEQ_LEN, SEQ_LEN):
    chunk = words[i : i + SEQ_LEN]
    corpus.append(chunk)

print(f"Tổng số chunks hợp lệ (64 tokens/chunk): {len(corpus)}")

Tổng số chunks hợp lệ (64 tokens/chunk): 3166


In [ ]:
# =========================
# 2. VOCABULARY & TOKENIZATION
# =========================

all_words = [word for chunk in corpus for word in chunk]
vocab_counts = Counter(all_words)

vocab = ['<pad>', '<unk>', '[CLS]', '[SEP]', '[MASK]'] + [w for w, c in vocab_counts.most_common()]

stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for i, w in enumerate(vocab)}
vocab_size = len(vocab)

print(f"Tổng số từ vựng (Vocab Size): {vocab_size}")

def encode(sentence_list):
    return [stoi.get(w, stoi['<unk>']) for w in sentence_list]

def decode(tokens):
    return " ".join([itos.get(t, '<unk>') for t in tokens])

Tổng số từ vựng (Vocab Size): 25670


In [ ]:
# =========================
# 3. MASKING LOGIC (MLM DATASET)
# =========================
# Thực hiện logic Mask 15% token

def create_mlm_data(sentence_tokens, mask_prob=0.15):
    input_ids = []
    labels = []

    for token in sentence_tokens:
        prob = random.random()
        if prob < mask_prob:
            prob /= mask_prob
            if prob < 0.8:
                input_ids.append(stoi['[MASK]'])
            elif prob < 0.9:
                input_ids.append(random.randint(5, vocab_size - 1))
            else:
                input_ids.append(token)

            labels.append(token)
        else:
            input_ids.append(token)
            labels.append(-100)

    return input_ids, labels

In [ ]:
# =========================
# 4. MODEL ARCHITECTURE (FIXED CONVERGENCE)
# =========================

class BidirectionalEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=300, num_heads=6, num_layers=4, max_seq_len=512, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim # Lưu lại số chiều
        self.token_emb = nn.Embedding(vocab_size, embed_dim, padding_idx=stoi['<pad>'])
        self.pos_emb = nn.Embedding(max_seq_len, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.lm_head = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0)

        emb = (self.token_emb(x) * math.sqrt(self.embed_dim)) + self.pos_emb(positions)

        padding_mask = (x == stoi['<pad>'])

        out = self.transformer(emb, src_key_padding_mask=padding_mask)
        logits = self.lm_head(out)
        return logits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BidirectionalEncoder(vocab_size).to(device)
print(model)

/tmp/ipykernel_589/3136233568.py:19: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


BidirectionalEncoder(
  (token_emb): Embedding(25670, 300, padding_idx=0)
  (pos_emb): Embedding(512, 300)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=300, out_features=300, bias=True)
        )
        (linear1): Linear(in_features=300, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=300, bias=True)
        (norm1): LayerNorm((300,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((300,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (lm_head): Linear(in_features=300, out_features=25670, bias=True)
)


In [ ]:
# =========================
# 5. DATA LOADER & BATCHING
# =========================
from torch.utils.data import DataLoader

def simple_collate_fn(batch_sentences):
    inputs_list = []
    labels_list = []

    for sentence in batch_sentences:
        encoded = encode(sentence)
        inp, lbl = create_mlm_data(encoded)

        inputs_list.append(inp)
        labels_list.append(lbl)

    inputs_tensor = torch.tensor(inputs_list)
    labels_tensor = torch.tensor(labels_list)

    return inputs_tensor, labels_tensor

BATCH_SIZE = 64
dataloader = DataLoader(
    corpus,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=simple_collate_fn
)

In [ ]:
# =========================
# 6. TRAINING LOOP (BATCH TRAINING)
# =========================

optimizer = optim.AdamW(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss(ignore_index=-100)
model.train()

EPOCHS = 500

for epoch in range(EPOCHS):
    total_loss = 0
    valid_batches = 0

    for batch_inputs, batch_labels in dataloader:
        batch_inputs = batch_inputs.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()

        logits = model(batch_inputs)

        loss = criterion(logits.view(-1, vocab_size), batch_labels.view(-1))

        if not torch.isnan(loss):
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            valid_batches += 1

    avg_loss = total_loss / valid_batches if valid_batches > 0 else 0

    if (epoch + 1) % 25 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Loss: {avg_loss:.4f}")

Epoch  25/500 | Loss: 8.4018
Epoch  50/500 | Loss: 7.1989
Epoch  75/500 | Loss: 6.6764
Epoch 100/500 | Loss: 6.1717
Epoch 125/500 | Loss: 5.7906
Epoch 150/500 | Loss: 5.4035
Epoch 175/500 | Loss: 5.1739
Epoch 200/500 | Loss: 4.9322
Epoch 225/500 | Loss: 4.7628
Epoch 250/500 | Loss: 4.5594
Epoch 275/500 | Loss: 4.3835
Epoch 300/500 | Loss: 4.2013
Epoch 325/500 | Loss: 4.0508
Epoch 350/500 | Loss: 3.9122
Epoch 375/500 | Loss: 3.7971
Epoch 400/500 | Loss: 3.7316
Epoch 425/500 | Loss: 3.5887
Epoch 450/500 | Loss: 3.5221
Epoch 475/500 | Loss: 3.3898
Epoch 500/500 | Loss: 3.3204


In [ ]:
# =========================
# 7. INFERENCE (SIMPLE CLOZE TASK)
# =========================
model.eval()

test_sentence = "So long and [MASK] for all the fish".split()

test_inputs = encode(test_sentence)

print("Tokens mô hình nhận được:", [itos.get(t, '<unk>') for t in test_inputs])

with torch.no_grad():
    inputs_tensor = torch.tensor([test_inputs]).to(device)
    logits = model(inputs_tensor)

    mask_idx = 3
    mask_logits = logits[0, mask_idx, :]

    special_tokens = ['<pad>', '<unk>', '[CLS]', '[SEP]', '[MASK]']
    for st in special_tokens:
        if st in stoi:
            mask_logits[stoi[st]] = -float('inf')

    top_3_idx = torch.topk(mask_logits, 3).indices.tolist()
    top_3_words = [itos[idx] for idx in top_3_idx]

print(f"Top 3 dự đoán cho [MASK]: {top_3_words}")

Tokens mô hình nhận được: ['So', 'long', 'and', '[MASK]', 'for', 'all', 'the', 'fish']
Top 3 dự đoán cho [MASK]: ['thanks', 'good', 'his']
